# Hierarchical Multi-Agent Deep Research & Report Planner
This notebook demonstrates a hierarchical multi-agent architecture using **pure LangChain (create_agent)** and **Google Gemini** for deep research and structured report generation.

### System Architecture:
```
                      ┌──────────────────┐
                      │   User Request   │
                      └────────┬─────────┘
                               │
                               ▼
                      ┌──────────────────┐
                      │    Main Agent    │
                      │ (Research Coord) │
                      └────────┬─────────┘
                               │
          ┌────────────────────┼────────────────────────┐
          │                    │                        │
          ▼                    ▼                        ▼
┌──────────────────┐  ┌──────────────────┐┌──────────────────┐  ┌──────────────────┐
│    Subagent A    │  │    Subagent B    ││    Subagent C    │  │    Subagent D    │
│ (Section Writer) │  │  (Intro Writer)  ││ (Conclusion Wr)  │  │ (Compiler)       │
└──────────────────┘  └──────────────────┘└──────────────────┘  └──────────────────┘
```

## 1. Setup & Environment Variables
Set up your API keys from `.env` or interactively.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load keys from root directory .env if present
if os.path.exists(".env"):
    load_dotenv(dotenv_path=".env")
elif os.path.exists("../.env"):
    load_dotenv(dotenv_path="../.env")
else:
    load_dotenv()

if "GEMINI_API_KEY" not in os.environ:
    if "GOOGLE_API_KEY" in os.environ:
        os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]
    else:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI API Key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API Key: ")

In [2]:
import prompt_inspector
prompt_inspector.inspect()

INFO:prompt-inspector:OpenAI client auto-patched successfully!
INFO:prompt-inspector:Anthropic client auto-patched successfully!
INFO:prompt-inspector:google-genai auto-patched successfully!


## 2. Research Tools
Define the Tavily search tool that our specialized Section Researcher subagent will use.

In [3]:
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

tavily_wrapper = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> str:
    """Searches the web for up-to-date information on a query."""
    try:
        results = tavily_wrapper.results(query, max_results=3)
        if not results:
            return f"No results found for query: '{query}'."
        
        output = f"Web Search Results for '{query}':\n"
        for i, res in enumerate(results, 1):
            output += f"Source {i}: {res.get('title', 'Untitled')}\n"
            output += f"URL: {res.get('url', 'N/A')}\n"
            output += f"Content: {res.get('content', '')}\n\n"
        return output
    except Exception as e:
        return f"Web search encountered an error: {str(e)}"

## 3. Subagent Instantiation
Instantiate the specialized subagents using `create_agent` from LangChain.

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# 1. Section Researcher & Writer
section_writer_agent = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt=(
        "You are a specialized Section Researcher & Writer Agent. Your job is to research "
        "and write a detailed section of a report in Markdown based on a given topic, section name, "
        "and description. Use the search_web tool to gather information. "
        "Write a highly technical, factual section of about 150-200 words. "
        "Start with a bold key insight, use short paragraphs, and list your sources at the end."
    )
)

# 2. Intro Writer
intro_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Introduction Writer Agent. Your job is to read "
        "completed body sections of a report and write a compelling Introduction "
        "(50-100 words, setting context). Do not call any tools, write directly based on the provided section drafts."
    )
)

# 3. Conclusion Writer
conclusion_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Conclusion Writer Agent. Your job is to read "
        "completed body sections of a report and write a structured Conclusion "
        "(100-150 words, synthesizing key findings and ending with next steps). "
        "Include a structured Markdown table summarizing the insights. Do not call any tools."
    )
)

# 4. Report Compiler
report_compiler_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Report Compiler Agent. Your job is to assemble all sections "
        "(Introduction, Body Sections, and Conclusion) into a single cohesive Wikipedia-style Markdown article.\n"
        "Structure it like a Wikipedia entry:\n"
        "1. Begin with a clean top-level heading (# [Topic]).\n"
        "2. Place a Wikipedia-style Infobox (as a Markdown table with bold keys) right at the beginning.\n"
        "3. Write a clear, concise introductory lead paragraph summarizing the topic before any subheadings.\n"
        "4. Do NOT write a Table of Contents manually; a python script will automatically inject it after the lead paragraph.\n"
        "5. Use clear, structured subheadings (## [Section] and ### [Subsection]).\n"
        "6. Format code terms using inline backticks.\n"
        "7. End with a structured 'References' section listing all sources cleanly.\n"
        "Ensure all special characters like '$' are escaped as '\\$' for correct rendering."
    )
)

## 4. Wrapping Subagents as Tools
To allow the main coordinator agent to invoke subagents, we wrap each subagent's execution pipeline in a LangChain `@tool`. We ensure `RunnableConfig` is propagated so that all subagent callbacks bubble up to the main stream.

In [5]:
from langchain_core.runnables import RunnableConfig
import datetime
import re
import os

def save_wikipedia_style_report(report_title, report_content):
    # Ensure title is stripped of Markdown formatting for filename
    safe_title = re.sub(r'[^a-zA-Z0-9\s\_\-]', '', report_title)
    safe_title = re.sub(r'\s+', '_', safe_title.strip())
    
    cleaned_content = report_content.strip()
    
    # Normalize top level title to # [report_title]
    if cleaned_content.startswith('#'):
        lines = cleaned_content.split('\n')
        orig_title = lines[0].replace('#', '').strip()
        if orig_title.lower().startswith('report:'):
            orig_title = orig_title[7:].strip()
        lines[0] = f'# {orig_title if orig_title else report_title}'
        cleaned_content = '\n'.join(lines)
    else:
        cleaned_content = f'# {report_title}\n\n{cleaned_content}'
        
    # Generate Table of Contents from h2 and h3
    headings = re.findall(r'^(#{2,3})\s+(.*)$', cleaned_content, re.MULTILINE)
    toc_lines = ['## Contents\n']
    for level_hashes, title in headings:
        indent = '  ' * (len(level_hashes) - 2)
        clean_title = re.sub(r'[\*\`\_]', '', title)
        anchor = clean_title.lower()
        anchor = re.sub(r'[^a-z0-9\s\-]', '', anchor)
        anchor = re.sub(r'\s+', '-', anchor)
        toc_lines.append(f'{indent}- [{clean_title}](#{anchor})')
    
    toc_str = '\n'.join(toc_lines) + '\n\n---\n'
    
    # Insert TOC before the first H2 subheading (after the lead paragraph)
    first_h2_match = re.search(r'^##\s+', cleaned_content, re.MULTILINE)
    if first_h2_match:
        idx = first_h2_match.start()
        final_content = cleaned_content[:idx] + toc_str + cleaned_content[idx:]
    else:
        # Prepend TOC after title
        lines = cleaned_content.split('\n')
        insert_idx = 1
        while insert_idx < len(lines) and not lines[insert_idx].strip():
            insert_idx += 1
        lines.insert(insert_idx, '\n' + toc_str)
        final_content = '\n'.join(lines)
        
    # Ensure reports folder exists
    os.makedirs('reports', exist_ok=True)
    report_path = f'reports/{safe_title}.md'
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(final_content)
        
    print(f'\n[Wikipedia-Style Export] Saved report to: {os.path.abspath(report_path)}')
    return final_content, report_path

@tool("Save_Wikipedia_Style_Report", description="Saves the final compiled report in a Wikipedia-style Markdown format to disk. Input should include the report_title (e.g. 'LangGraph State Management') and the full report_content.")
def call_save_wikipedia_report(report_title: str, report_content: str) -> str:
    """Saves the final report to disk in a Wikipedia-style Markdown format."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Save_Wikipedia_Style_Report (Title: {report_title})")
    try:
        wiki_content, report_path = save_wikipedia_style_report(report_title, report_content)
        now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"\n[{now}] <<< EXIT: Save_Wikipedia_Style_Report finished")
        return f"Success: Wikipedia-style report saved successfully to {report_path}."
    except Exception as e:
        now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"\n[{now}] <<< EXIT: Save_Wikipedia_Style_Report failed: {str(e)}")
        return f"Error saving report: {str(e)}"

@tool("Section_Researcher_Writer", description="Researches and writes a single section of a report. Input should be a query containing the section name, description, and overall topic.")
def call_section_writer(query: str, config: RunnableConfig) -> str:
    """Call the section researcher subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Section_Researcher_Writer (Query: {query[:60]}...)")
    result = section_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Section_Researcher_Writer finished")
    return result['messages'][-1].content

@tool("Intro_Writer", description="Drafts the Introduction section. Input must include the overall topic and all completed body sections as context.")
def call_intro_writer(query: str, config: RunnableConfig) -> str:
    """Call the intro subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Intro_Writer")
    result = intro_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Intro_Writer finished")
    return result['messages'][-1].content

@tool("Conclusion_Writer", description="Drafts the Conclusion section. Input must include the overall topic and all completed body sections as context.")
def call_conclusion_writer(query: str, config: RunnableConfig) -> str:
    """Call the conclusion subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Conclusion_Writer")
    result = conclusion_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Conclusion_Writer finished")
    return result['messages'][-1].content

@tool("Report_Compiler", description="Assembles and compiles the final report structure. Input must contain the introduction, all body sections, and the conclusion to format and combine.")
def call_report_compiler(query: str, config: RunnableConfig) -> str:
    """Call the report compiler subagent."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Report_Compiler")
    result = report_compiler_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Report_Compiler finished")
    return result['messages'][-1].content

main_agent_tools = [call_section_writer, call_intro_writer, call_conclusion_writer, call_report_compiler, call_save_wikipedia_report]

## 5. Main Coordinator Agent
Set up the Main Agent (Deep Research Planner) with detailed system instructions guiding it through planning, delegation, and final synthesis. To achieve parallelization, we instruct the coordinator to trigger multiple tool calls concurrently in a single step.

In [6]:
main_agent_prompt = (
    "You are the Deep Research Planner, a main coordinator agent. "
    "Your goal is to coordinate a deep research report generation on a topic requested by the user. "
    "\\n\\n"
    "Available subagents and tools:\\n"
    "1. Section_Researcher_Writer: Researches and writes a single section of the report.\\n"
    "2. Intro_Writer: Drafts the Introduction using the body sections.\\n"
    "3. Conclusion_Writer: Drafts the Conclusion using the body sections.\\n"
    "4. Report_Compiler: Combines, formats, and compiles all sections into a single cohesive Markdown report.\\n"
    "5. Save_Wikipedia_Style_Report: Saves the final compiled report in a Wikipedia-style Markdown format to disk.\\n"
    "\\n\\n"
    "Step-by-step workflow:\\n"
    "a. Determine the structure of the report (e.g., plan 2-3 main body sections based on the user's topic).\\n"
    "b. Call Section_Researcher_Writer FOR ALL PLANNED SECTIONS IN PARALLEL (trigger multiple tool calls in a single turn/step) to write all body sections concurrently.\\n"
    "c. Once the body sections are written, call BOTH Intro_Writer and Conclusion_Writer IN PARALLEL (trigger tool calls for both in a single turn/step) to write the introduction and conclusion concurrently.\\n"
    "d. Call Report_Compiler with the Intro, the drafted sections, and the Conclusion to build the compiled report.\\n"
    "e. Call Save_Wikipedia_Style_Report with the report title (e.g., 'LangGraph State Management') and the compiled report content to save it to disk.\\n"
    "f. Output the final compiled report and the tool confirmation message to the user.\\n\\n"
    "CRITICAL: ALWAYS invoke parallel tool calls together in the same step to achieve concurrency. Do not call them one by one."
)

main_agent = create_agent(
    model=llm,
    tools=main_agent_tools,
    system_prompt=main_agent_prompt
)

## 6. Execution Run
Let's test the entire multi-agent system with a request. Watch it plan, delegate in parallel, and compile!

In [7]:
user_request = (
    "I want a research report on 'Multiverse'."
    "What is the current state of research on the Multiverse? Include sections on theoretical foundations, observational evidence, and implications for physics and cosmology."
)

print("=== STARTING DEEP RESEARCH COORDINATOR AGENT ===\\n")
result = main_agent.invoke({"messages": [{"role": "user", "content": user_request}]})
print("\\n=== FINAL REPORT RESPONSE ===\\n")
print(result["messages"][-1].content[0]['text'] if isinstance(result["messages"][-1].content, list) else result["messages"][-1].content)

=== STARTING DEEP RESEARCH COORDINATOR AGENT ===\n


DEBUG wrapped_generate (new SDK):
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):



[20:37:15.863] >>> ENTER: Section_Researcher_Writer (Query: Theoretical foundations of the Multiverse, including inflati...)

[20:37:15.865] >>> ENTER: Section_Researcher_Writer (Query: Observational evidence and challenges in detecting the Multi...)

[20:37:15.867] >>> ENTER: Section_Researcher_Writer (Query: Implications of the Multiverse for physics and cosmology, in...)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"



[20:37:21.932] <<< EXIT: Section_Researcher_Writer finished


...


[20:37:36.294] <<< EXIT: Report_Compiler finished


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):



[20:37:40.312] >>> ENTER: Save_Wikipedia_Style_Report (Title: Multiverse Research Report)

[Wikipedia-Style Export] Saved report to: /Users/sachinmishra/Desktop/Agents_From_Scratch/Langchain_Agents/reports/Multiverse_Research_Report.md

[20:37:40.316] <<< EXIT: Save_Wikipedia_Style_Report finished


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


\n=== FINAL REPORT RESPONSE ===\n
The research report on the 'Multiverse' has been successfully compiled and saved. You can find the full report below:

# Multiverse

| Key | Information |
| :--- | :--- |
| **Concept** | Hypothetical set of multiple observable universes |
| **Primary Fields** | Cosmology, String Theory, Quantum Mechanics |
| **Key Mechanisms** | Eternal Inflation, String Landscape, Many-Worlds Interpretation |
| **Status** | Theoretical framework; currently unverified |

The concept of the multiverse challenges our fundamental understanding of reality, suggesting that our observable universe is merely one of countless distinct domains. This report explores the theoretical foundations of this hypothesis, drawing on inflation, string theory, and quantum mechanics to explain how such vast structures might arise. By examining potential observational evidence, including cosmic microwave background anomalies and theoretical bubble collisions, we investigate the validity of t